In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder\
.appName("PysparkAssessment1")\
.getOrCreate()

spark

In [ ]:
# Data Ingestion & Exploration
# Load both CSV files with schema inference.
from google.colab import drive
drive.mount('/content/drive')

customer_df = spark.read.csv('/content/drive/MyDrive/customers.csv',header= True,inferSchema=True)
orders_df = spark.read.csv('/content/drive/MyDrive/orders.csv',header= True,inferSchema=True)
customer_df.show(5)
orders_df.show(5)

In [ ]:
# List all columns and data types.

print("Customers Schema:")
customer_df.printSchema()

print("Orders Schema:")
orders_df.printSchema()

In [ ]:
# Count the total number of customers and orders.

total_customers = customer_df.count()
total_orders = orders_df.count()

print("Total number of customers:",total_customers)
print("Total number of orders:",total_orders)

In [ ]:
# Show distinct cities.

dis_city = customer_df.select("City").distinct()
dis_city.show()

In [ ]:
# DataFrame Transformations
# Add a column TotalAmount = Price * Quantity

orders_df = orders_df.withColumn("TotalAmount",orders_df["Price"] * orders_df["Quantity"])
orders_df.show()

In [ ]:
# Create a new column OrderYear from OrderDate .

from pyspark.sql.functions import year

orders_df = orders_df.withColumn("OrderYear",year(orders_df['OrderDate']))
orders_df.show()

In [ ]:
# Filter orders with TotalAmount > 10,000 .

gratervalue = orders_df.filter(orders_df['TotalAmount'] > 10000)
gratervalue.show()

In [ ]:
# Drop the Email column from customers .

customer_df = customer_df.drop('Email')
customer_df.show()

In [ ]:
# Handling Nulls & Conditionals
# Simulate a null in City and fill it with ?Unknown?.

from pyspark.sql.functions import when

customer_df = customer_df.withColumn("City",when(customer_df['CustomerID'] == 103,None).otherwise(customer_df["City"]))

customer_df = customer_df.fillna({"City":"Unknown"})
customer_df.show()

In [ ]:
# Label customers as ?Loyal? if SignupDate is before 2022, else ?New?.

from pyspark.sql.functions import when

customer_df = customer_df.withColumn("CustomerLable",when(customer_df["SignupDate"] < "2022-01-01","Loyal").otherwise("New"))
customer_df.show()

In [ ]:
# Create OrderType column: "Low" if < 5,000, "High" if >5,000.

from pyspark.sql.functions import when

orders_df = orders_df.withColumn("OrderType",when(orders_df['Price'] < 5000,"Low").otherwise("High"))
orders_df.show()

In [ ]:
# Joins & Aggregations
# Join customers and orders on CustomerID .

joined_df = orders_df.join(customer_df,on = "CustomerID",how = "inner")
joined_df.show()

In [ ]:
# Get total orders and revenue per city.
from pyspark.sql.functions import count, sum

city_total = joined_df.groupBy("City").agg(count("OrderID"),sum("TotalAmount"))
city_total.show()

In [ ]:
# Show top 3 customers by total spend.

from pyspark.sql.functions import col

top_3_customer = joined_df.groupBy("CustomerID","Name").agg(sum("TotalAmount").alias("TotalSpent")).orderBy(col("TotalSpent").desc()).limit(3)
top_3_customer.show()

In [ ]:
# Count how many products each category has sold.

from pyspark.sql.functions import sum

category_sold = joined_df.groupBy("Category").agg(sum("Quantity").alias("TotalProductSold"))
category_sold.show()

In [ ]:
# Spark SQL Tasks
# Create database sales and switch to it.

spark.sql("create database sales")

In [ ]:
# Save both datasets as tables in the sales database.

customer_df.write.mode("overwrite").saveAsTable("sales.customers")
orders_df.write.mode("overwrite").saveAsTable("sales.orders")

In [ ]:
# Write SQL to:
# List all orders by customers from ?Delhi?.
spark.sql("""select o.*
from sales.orders o
join sales.customers c
on o.CustomerID = c.CustomerID
where c.city = 'Delhi'""").show()
# Find average order value in each category.
spark.sql("""select category,avg(totalamount) from sales.orders group by category""").show()
# Create a view monthly_orders with month-wise total amount.
spark.sql("""create or replace temp view monthly_orders as
select date_format(OrderDate,'yyyy-mm') as month,
sum(price*quantity) as TotalAmount from sales.orders
group by date_format(OrderDate,'yyyy-mm')""")

spark.sql("SELECT * FROM monthly_orders").show()

In [ ]:
from os import truncate
# String & Date Functions
# Mask emails using regex (e.g., a***@gmail.com ).

from pyspark.sql.functions import regexp_replace

masked_email = customer_df.withColumn("MaskedEmail",regexp_replace("Email",r"(^.)([^@]+)(@.*)", r"\1***\3"))
masked_email.show(truncate = False)

In [ ]:
# Concatenate Name and City as ?Name from City?.

from pyspark.sql.functions import concat_ws

name_city = customer_df.withColumn("NameWithColumn",concat_ws("from",customer_df["Name"],customer_df["City"]))
name_city.show()

In [ ]:
# Use datediff() to calculate customer age in days.

from pyspark.sql.functions import current_date,date_diff

customer_age_diff = customer_df.withColumn("AgeInDays",date_diff(current_date(),customer_df["SignupDate"]))
customer_age_diff.show()

In [ ]:
# Extract month name from OrderDate .

from pyspark.sql.functions import date_format

orders_with_month = orders_df.withColumn("OrderMonth",date_format("OrderDate","MMMM"))
orders_with_month.show()

In [ ]:
# UDFs and Complex Logic
# Write a UDF to tag customers:
# ?Gold? if spend >50K, ?Silver? if 10K?50K, ?Bronze? if <10K.

from pyspark.sql.functions import udf, sum, col
from pyspark.sql.types import StringType

def tag_customer(spend):
  if spend > 50000:
    return "Gold"
  elif spend >= 10000:
    return "Silver"
  else:
    return "Bronze"

tag_udf = udf(tag_customer,StringType())

# Write a UDF to shorten product names (first 3 letters + ...).

customer_spend_df = orders_df.withColumn("Spend", col("Price") * col("Quantity")) \
    .groupBy("CustomerID").agg(sum("Spend").alias("TotalSpend"))

customer_tagged_df = customer_spend_df.withColumn("Tier", tag_udf("TotalSpend"))
customer_tagged_df.show()

In [ ]:
# Parquet & Views
# Save the joined result as a Parquet file.
joined_df.write.mode("overwrite").parquet("output/joined_data.parquet")

In [ ]:
# Read it back and verify schema.
parquet_df = spark.read.parquet("output/joined_data.parquet")

parquet_df.printSchema()
parquet_df.show(truncate = False)

In [ ]:
# Create and query a global temp view.
parquet_df.createOrReplaceGlobalTempView("join_order_customer")

spark.sql("""SELECT Name, Product, TotalAmount, City
FROM global_temp.join_order_customer
WHERE TotalAmount > 10000""").show()

In [ ]:
# Compare performance between CSV read and Parquet read.

import time

# Time CSV read
start_csv = time.time()
csv_df = spark.read.option("header", True).option("inferSchema", True).csv("/content/drive/MyDrive/orders.csv")
csv_df.count()
end_csv = time.time()

# Time Parquet read
start_parquet = time.time()
parquet_df = spark.read.parquet("output/joined_data.parquet")
parquet_df.count()
end_parquet = time.time()

print(f"CSV Read Time: {end_csv - start_csv:.4f} seconds")
print(f"Parquet Read Time: {end_parquet - start_parquet:.4f} seconds")